In [15]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/sephora_clean.csv")
df_clean.head(5)

/var/folders/bx/xxftvqk17n33yx6l437wb7200000gn/T/ipykernel_2817/657677713.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv("../data/processed/sephora_clean.csv")


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,...,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,rating_category
0,21907641410,3.0,1.0,0.0,1.0,1.0,0.0,2023-02-25,nice consistency however im not sure its doing...,Not sure it’s doing what was promised,...,0,0,0,0,NaN,Skincare,Eye Care,Eye Creams & Treatments,0,neutral
1,5201105842,5.0,1.0,1.0,1.0,0.0,1.0,2023-02-16,i cannot express how much i love this eye crea...,Genuinely amazing,...,0,0,0,0,NaN,Skincare,Eye Care,Eye Creams & Treatments,0,positive
2,1878596139,5.0,1.0,1.0,2.0,0.0,2.0,2022-10-31,this depuffs super well does not help with dar...,Excellent de-puffer!,...,0,0,0,0,NaN,Skincare,Eye Care,Eye Creams & Treatments,0,positive
3,1949686936,4.0,1.0,1.0,2.0,0.0,2.0,2022-06-13,love it it keeps crows feet and fine lines awa...,NaN,...,0,0,0,0,NaN,Skincare,Eye Care,Eye Creams & Treatments,0,positive
4,5651999467,5.0,1.0,1.0,2.0,0.0,2.0,2022-06-11,im 46 and i can honestly say this has helped m...,really helps relax and blur the lines.,...,0,0,0,0,NaN,Skincare,Eye Care,Eye Creams & Treatments,0,positive


In [16]:
df_nlp = df_clean.copy()

df_nlp["review_title"] = df_nlp["review_title"].fillna("")
df_nlp["review_text"] = df_nlp["review_text"].fillna("")

df_nlp["text"] = df_nlp["review_title"] + " " + df_nlp["review_text"]
df_nlp["text"] = df_nlp["text"].str.strip()

In [17]:
df_nlp = df_nlp[df_nlp["text"].str.len() > 0].copy()

In [18]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)      # linkleri kaldır
    text = re.sub(r"[^a-z\s]", " ", text)            # sadece harfler
    text = re.sub(r"\s+", " ", text).strip()         # fazla boşlukları temizle
    return text

df_nlp["clean_text"] = df_nlp["text"].apply(clean_text)

In [19]:
# 1. MODEL İÇİN GEREKLİ KOLONLARI SEÇ
df_model = df_nlp[[
    "product_id",
    "brand_name",
    "product_name",
    "primary_category",
    "secondary_category",
    "clean_text",
    "rating",
    "price_usd",
    "rating_category"
]].copy()

# sadece positive / negative bırak
df_model = df_model[df_model["rating_category"].isin(["positive", "negative"])].copy()

df_model.head()

,product_id,brand_name,product_name,primary_category,secondary_category,clean_text,rating,price_usd,rating_category
1,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,genuinely amazing i cannot express how much i ...,5.0,89.0,positive
2,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,excellent de puffer this depuffs super well do...,5.0,89.0,positive
3,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,love it it keeps crows feet and fine lines awa...,4.0,89.0,positive
4,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,really helps relax and blur the lines im and i...,5.0,89.0,positive
5,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,it was ok but no different frommy other cream ...,2.0,89.0,negative


In [20]:
# 2. SENTIMENT MODELİ KUR
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = df_model["clean_text"]
y = df_model["rating_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model', LogisticRegression(max_iter=1000))])

In [21]:
# 3. MODEL PERFORMANSINA BAK
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.89      0.79      0.84     22810
    positive       0.97      0.99      0.98    179389

    accuracy                           0.97    202199
   macro avg       0.93      0.89      0.91    202199
weighted avg       0.97      0.97      0.97    202199



In [22]:
# 4. TÜM VERİYE SENTIMENT TAHMİNİ YAP
df_model["sentiment_pred"] = pipeline.predict(df_model["clean_text"])

In [23]:
# 5. SENTIMENT SCORE ÜRET
# predict_proba çıktısında kolon sırası classes_ ile belirlenir
print(pipeline.named_steps["model"].classes_)

['negative' 'positive']


In [24]:
df_model["sentiment_score"] = pipeline.predict_proba(df_model["clean_text"])[:, 1]

In [25]:
df_model.head(5)

,product_id,brand_name,product_name,primary_category,secondary_category,clean_text,rating,price_usd,rating_category,sentiment_pred,sentiment_score
1,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,genuinely amazing i cannot express how much i ...,5.0,89.0,positive,positive,0.999185
2,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,excellent de puffer this depuffs super well do...,5.0,89.0,positive,positive,0.982304
3,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,love it it keeps crows feet and fine lines awa...,4.0,89.0,positive,positive,0.998770
4,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,really helps relax and blur the lines im and i...,5.0,89.0,positive,positive,0.998103
5,P107306,Murad,Renewing Eye Cream,Skincare,Eye Care,it was ok but no different frommy other cream ...,2.0,89.0,negative,negative,0.243457


In [26]:
# 6. İSTEDİĞİN SON TABLOYU OLUŞTUR
product_summary = df_model.groupby(
    ["primary_category","secondary_category", "product_id", "brand_name", "product_name"],
    as_index=False
).agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count"),
    avg_sentiment=("sentiment_score", "mean"),
    positive_ratio=("sentiment_pred", lambda x: (x == "positive").mean()),
    negative_ratio=("sentiment_pred", lambda x: (x == "negative").mean()),
    price=("price_usd", "mean")
)

product_summary.head()

,primary_category,secondary_category,product_id,brand_name,product_name,avg_rating,rating_count,avg_sentiment,positive_ratio,negative_ratio,price
0,Skincare,Cleansers,P122651,CLINIQUE,Clarifying Lotion 1,4.594737,190,0.911736,0.936842,0.063158,20.0
1,Skincare,Cleansers,P122661,CLINIQUE,7 Day Face Scrub Cream Rinse-Off Formula,4.596078,765,0.927823,0.939869,0.060131,26.0
2,Skincare,Cleansers,P122718,CLINIQUE,Exfoliating Face Scrub,4.653747,1161,0.938107,0.955211,0.044789,26.0
3,Skincare,Cleansers,P122762,CLINIQUE,Rinse-Off Foaming Cleanser,4.565773,783,0.915807,0.934866,0.065134,23.5
4,Skincare,Cleansers,P122876,CLINIQUE,Clarifying Lotion 4,4.512077,207,0.902242,0.927536,0.072464,20.0


In [27]:
product_summary.to_csv("../data/processed/nlp_product_summary.csv", index=False)